# MNIST evaluation notebook (SNN vs MC Dropout vs EDL)

Thin notebook wrapper around `snn_eval.run_mnist`. Every training / inference / metrics / plotting step below calls `run_mnist.compute()` and `run_mnist.display()` directly — nothing here re-implements that logic, so any fix made in `run_mnist.py` is picked up automatically next time this notebook runs.

Section 2 shows the one extension point (`compute(args, loader=..., exp_name=...)`) that lets this same pipeline run against another MNIST-shaped dataset without touching training/inference/metrics code.

In [ ]:
%matplotlib inline
import argparse
import pandas as pd
from snn_eval import run_mnist, cache

In [ ]:
def run_cached(exp_name, args, loader=None):
    """Thin caching wrapper around run_mnist.compute(), mirroring run_mnist.main().

    Avoids retraining / re-running inference on every notebook re-run: results
    are keyed by all params (like the CLI's results cache) and models by the
    train-only subset (like the CLI's model cache, handled inside compute()
    itself). Delete results/cache or results/models to force a clean re-run.
    """
    params = {k: v for k, v in vars(args).items() if k not in ("device", "no_cache")}
    res = None if args.no_cache else cache.load_results(exp_name, params)
    if res is None:
        res = run_mnist.compute(args, loader=loader, exp_name=exp_name)
        cache.save_results(exp_name, params, res)
    return res

## 1. MNIST (default loader)

`build_argparser()` is the exact same parser `run_mnist.py` uses from the CLI, so `parse_args([])` gives a `Namespace` of CLI defaults — tweak individual fields below instead of re-typing them.

In [ ]:
args = run_mnist.build_argparser().parse_args([])  # identical defaults to the CLI
args.arch = "mlp"  # "mlp" | "cnn" | "resnet"

QUICK = True  # flip to False for the full paper-grade run
if QUICK:
    args.epochs, args.Np, args.Nm, args.T = 2, 3, 3, 20
    args.rot_step, args.rot_n = 45, 200

vars(args)

In [ ]:
res = run_cached("run_mnist", args)

In [ ]:
fig = run_mnist.display(res)  # prints tables + sweep, saves results/rotation_sweep.{csv,png}, shows inline

In [ ]:
pd.DataFrame(res["table"])

## 2. Adapting to another dataset

`compute()` accepts a `loader` — any zero-arg callable returning `((Xtr, ytr), (Xte, yte), Xood)` with images shaped `(N, 1, 28, 28)` and integer labels, exactly like `run_mnist.load_mnist`. Pass a different one and the rest of the pipeline (model build, training, nested sampling, MC Dropout, EDL, metrics, rotation sweep, plotting) is untouched — `compute()` now infers `K` from the labels rather than hardcoding 10.

Use a distinct `exp_name` so the alternate run's model/result cache doesn't collide with the MNIST one, and a distinct `out_prefix` in `display()` so the CSV/PNG don't overwrite `rotation_sweep.*`.

**Scope caveat:** the CNN/ResNet backbones in `models.py` (`_CNNBackbone`, `_TinyResNet`) are hardcoded for single-channel 28×28 input, so this loader-swap trick is only valid for other MNIST-shaped datasets (FashionMNIST, KMNIST, EMNIST, ...). For a genuinely different shape/channel-count/class-count (e.g. CIFAR-10), use the repo's other evaluation-ladder rungs instead — `run_exp1.py` (frozen DINOv2/CLIP backbone) or `train_fullnet.py` (full ResNet18/WRN-28-10) — rather than forcing this MNIST pipeline.

In [ ]:
def load_fashion_as_id(root="./data", ntr=60000, nte=10000):
    """Same return contract as run_mnist.load_mnist, but FashionMNIST is ID and
    MNIST is OOD (roles swapped). Swap the dataset names for KMNIST/EMNIST etc.
    """
    import torch
    import torchvision as tv, torchvision.transforms as T
    tf = T.ToTensor()
    tr  = tv.datasets.FashionMNIST(root, train=True,  download=True, transform=tf)
    te  = tv.datasets.FashionMNIST(root, train=False, download=True, transform=tf)
    ood = tv.datasets.MNIST(root, train=False, download=True, transform=tf)
    def imgs(ds, n):
        X = torch.stack([ds[i][0] for i in range(min(n, len(ds)))])
        y = torch.tensor([ds[i][1] for i in range(min(n, len(ds)))])
        return X, y
    return imgs(tr, ntr), imgs(te, nte), imgs(ood, nte)[0]

In [ ]:
args_fashion = argparse.Namespace(**vars(args))  # same knobs, incl. QUICK tweaks

res_fashion = run_cached("fashion_mnist", args_fashion, loader=load_fashion_as_id)
fig_fashion = run_mnist.display(res_fashion, out_prefix="fashion_rotation_sweep")

## 3. Compare datasets side by side

Both `res` dicts came from the same `compute()`, so the table rows line up directly — no metric recomputation here, just reshaping already-computed numbers.

In [ ]:
t1 = pd.DataFrame(res["table"]).assign(dataset="MNIST")
t2 = pd.DataFrame(res_fashion["table"]).assign(dataset="FashionMNIST(ID)/MNIST(OOD)")
pd.concat([t1, t2]).set_index(["dataset", "name"])